## Install psycopg2 (Postgre DB connector)

In [2]:
%pip install psycopg2-binary
%pip install psycopg2

# NOTE: Installing psycopg2 in ArcGIS Pro is non-trivial.
# Regular `pip install` installs into the system Python, not ArcGIS's env.
# 
# The only working approach:
#   1. Open Command Prompt as Administrator
#   2. Run: "C:\Program Files\ArcGIS\Pro\bin\Python\Scripts\conda.exe"
#            install -n arcgispro-py3 -c conda-forge psycopg2 --force-reinstall -y

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Import used libraries

In [1]:
import arcpy
import pandas as pd
import numpy as np
import psycopg2
import os
import tempfile

### Read the constructions_locations data from the Postgre database

In [2]:
db_connection = psycopg2.connect(
    host="http://46.225.163.131",  # IpV6: 2a01:4f8:c2c:97ba::1 (Have in mind that the DB port isn't opened anymore)
    port=5432,
    dbname="construction_coords",
    user="postgres",
    password="PowerCell46"
)

database_result_construction_coordinates = pd.read_sql(
    "SELECT * FROM constructions_locations", 
    db_connection
)

db_connection.close()

database_result_construction_coordinates

C:/Users/HPZBOO~1/AppData/Local/Temp/ArcGISProTemp17668/xpython_17668/706395676.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  database_result_construction_coordinates = pd.read_sql(


,id,created_at,latitude,longitude
0,51259417-3a6d-4a8e-b7f3-f6526377e4bc,2026-04-14 20:52:11.452502,42.679150,23.311926
1,bc1da7a3-2da9-44b9-bda4-5494973d92cd,2026-04-14 20:53:29.291918,42.687853,23.316220
2,24853d18-5980-4c89-b6e5-69407d1aea8c,2026-04-14 20:54:59.441805,42.664786,23.297083
3,7fafdd44-1312-4bbe-b39f-69b9ccc248e9,2026-04-14 21:12:16.053456,42.690997,23.310766
4,44915c04-da0f-4e94-addc-e7342186956b,2026-04-15 07:36:14.553909,42.691959,23.314711
5,ef0ef385-a6ed-4067-aa9e-0dc9e03ab898,2026-04-15 09:19:36.304594,42.681216,23.309541
6,de9944fa-867a-4471-9af5-947eef64fbc2,2026-04-15 09:34:56.451446,42.680770,23.431230
7,1891cdf0-5eb9-43e5-8fe1-1982bfb8414b,2026-04-15 09:35:03.259736,42.680770,23.431230
8,40f58ba0-6e7a-477e-8d82-0bbf37c45db2,2026-04-15 09:43:21.873126,42.680770,23.431230


### Create a constructions_points layer with the data from the previous step

In [3]:
WGS84_CODE = 4326
UTM_35N_CODE = 32635

# CURRENT_WORKING_DIR_PATH = os.getcwd()
CURRENT_WORKING_DIR_PATH = "C:\\Users\\HP ZBook 17 G5\\Documents\\ArcGIS\\Projects\\HandsOnTrainingWithPython"
GEODATABASE_PATH = os.path.join(CURRENT_WORKING_DIR_PATH, "HandsOnTrainingWithPython.gdb")

# Temp CSV with the read data from Postgre
temp_csv = os.path.join(CURRENT_WORKING_DIR_PATH, "temp_coords.csv")
database_result_construction_coordinates.to_csv(temp_csv, index=False)

arcpy.management.XYTableToPoint(
    in_table=temp_csv,
    out_feature_class=os.path.join(GEODATABASE_PATH, "Construction_points_WGS84"),
    x_field="longitude",  # x = longitude
    y_field="latitude",  # y = latitude
    z_field=None,
    coordinate_system=arcpy.SpatialReference(WGS84_CODE)
)

# Create a new layer with the correct projection
arcpy.management.Project(
    in_dataset=os.path.join(GEODATABASE_PATH, "Construction_points_WGS84"),
    out_dataset=os.path.join(GEODATABASE_PATH, "Construction_points_UTM35N"),
    out_coor_system=arcpy.SpatialReference(UTM_35N_CODE)
)

os.remove(temp_csv)
# Delete the layer with WGS84 projection, we don't need it
arcpy.management.Delete(os.path.join(GEODATABASE_PATH, "Construction_points_WGS84"))

<Result 'true'>

### Calculate new land_area_name column, based on the point's position

In [4]:
construction_points_layer = os.path.join(GEODATABASE_PATH, "Construction_points_UTM35N")

arcpy.management.AddField(
    construction_points_layer,
    "land_area_name",
    "TEXT",
    field_alias="Име на землището",
    field_length=150
)

land_areas_layer = os.path.join(GEODATABASE_PATH, "Bg_Land_areas")

arcpy.analysis.SpatialJoin(
    target_features=construction_points_layer,
    join_features=land_areas_layer,
    out_feature_class="temp_join",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"  # Don't use WITHIN (must be fully inside the polygon. Points that fall exactly on a polygon boundary are excluded) [we have such case]
)

lookup = {}
with arcpy.da.SearchCursor("temp_join", ["TARGET_FID", "Name_bg"]) as cursor:
    for row in cursor:
        lookup[row[0]] = row[1]

with arcpy.da.UpdateCursor(construction_points_layer, ["OID@", "land_area_name"]) as cursor:
    for row in cursor:
        row[1] = lookup.get(row[0])
        cursor.updateRow(row)  # TODO: Can't we bulk update? (sounds slow)

# Delete the temporary layer, we don't need it anymore
arcpy.management.Delete("temp_join")

<Result 'true'>

### Calculate new distance_to_road_meters column, based on how far a point is from the nearest road

In [9]:
arcpy.management.AddField(
    construction_points_layer,
    "distance_to_road_meters",
    "Double",
    field_alias="Дистанция до път (метри)",
    field_scale=3
)

roads_layer = os.path.join(GEODATABASE_PATH, "Bg_Roads")

arcpy.analysis.Near(
    in_features=construction_points_layer,
    near_features=roads_layer,
    search_radius=None,
    location="NO_LOCATION",
    angle="NO_ANGLE",
    method="PLANAR"
)

with arcpy.da.UpdateCursor(construction_points_layer, ["NEAR_DIST", "distance_to_road_meters"]) as cursor:
    for row in cursor:
        row[1] = row[0]
        cursor.updateRow(row)

arcpy.management.DeleteField(construction_points_layer, ["NEAR_DIST", "NEAR_FID"])

<Result 'C:\\Users\\HP ZBook 17 G5\\Documents\\ArcGIS\\Projects\\HandsOnTrainingWithPython\\HandsOnTrainingWithPython.gdb\\Construction_points_UTM35N'>